# Exploring the Dataset: Building the Hosts Table

**Goal:** Understand how we go from a raw LOG configuration file to a relational database table.

This notebook walks through:
1. Loading `auth.log` (contains access authorization logs to the intranet server)
2. Examining what fields each access log
3. Transforming the data into a Pandas DataFrame

---

**Dataset:** AIT Log Data Set V2.0 — russellmitchell testbed  
**Source:** https://zenodo.org/records/5789064

## 0. Configuration

Set the path to the `russellmitchell/` dataset folder below.  

**Default:** Assumes `russellmitchell/` is at the same level as the repo:
```
data-201-group-project/
├── data-201-security-log-analysis/   <-- this repo
│   └── notebooks/                    <-- this notebook is here
└── russellmitchell/                  <-- dataset is here
```

If your dataset is somewhere else, just change `DATASET_ROOT` below.

In [ ]:
import re
from pathlib import Path

import pandas as pd

# --- CHANGE THIS if your dataset is in a different location ---
DATASET_ROOT = Path("..") / ".." / "russellmitchell"

# Verify the path exists
if not DATASET_ROOT.exists():
    print(f"ERROR: Dataset not found at: {DATASET_ROOT.resolve()}")
    print("")
    print("Expected directory structure:")
    print("  <workspace>/data-201-security-log-analysis/notebooks/  <-- you are here")
    print("  <workspace>/russellmitchell/                          <-- dataset should be here")
    print("")
    print("Fix: Update DATASET_ROOT above to point to your russellmitchell/ folder.")
else:
    print(f"Dataset found at: {DATASET_ROOT.resolve()}")

Dataset found at: /Users/rrosevearehunt/PythonProjects/DATA201/GroupProject/russellmitchell


## 1. Load the Raw log File

The file `gather/intranet_server/logs/auth.log` 

In [ ]:
auth_log = DATASET_ROOT / "gather" / "intranet_server" / "logs" / "auth.log"


def parse_auth_log(file_path):
    # Regular expression to match the standard syslog date/time format
    # Group 1: Month (e.g., Jan)
    # Group 2: Day (e.g., 23)
    # Group 3: Time (e.g., 06:25:05)
    # Group 4: The rest of the log message
    log_pattern = re.compile(r"^([A-Z][a-z]{2})\s+(\d+)\s+(\d{2}:\d{2}:\d{2})\s+(.*)$")

    parsed_data = []
    with open(file_path) as file:
        for line in file:
            line = line.strip()

            # Ignore empty lines or metadata tags (like ) if present
            if not line or line.startswith("["):
                continue

            match = log_pattern.search(line)
            if match:
                month, day, time, log_message = match.groups()
                parsed_data.append({"Month": month, "Day": day, "Time": time, "Log": log_message})

    # Convert the list of dictionaries into a pandas DataFrame
    df = pd.DataFrame(parsed_data)
    return df


# Execute the function assuming the file is named 'auth.log' in the same directory
log_table = parse_auth_log(auth_log)

if log_table is not None:
    # Set pandas display options to show the full log message without truncating
    pd.set_option("display.max_colwidth", None)

    # Display the first 10 rows of the table
    print(log_table.head(20))

   Month Day      Time  \
0    Jan  23  06:25:05   
1    Jan  23  06:39:01   
2    Jan  23  06:39:01   
3    Jan  23  06:47:01   
4    Jan  23  06:47:02   
5    Jan  23  07:07:01   
6    Jan  23  07:07:01   
7    Jan  23  07:09:01   
8    Jan  23  07:09:01   
9    Jan  23  07:17:01   
10   Jan  23  07:17:01   
11   Jan  23  07:39:01   
12   Jan  23  07:39:01   
13   Jan  23  08:09:01   
14   Jan  23  08:09:01   
15   Jan  23  08:17:01   
16   Jan  23  08:17:01   
17   Jan  23  08:39:01   
18   Jan  23  08:39:01   
19   Jan  23  09:09:01   

                                                                                             Log  
0              intranet-server CRON[22883]: pam_unix(cron:session): session closed for user root  
1   intranet-server CRON[23064]: pam_unix(cron:session): session opened for user root by (uid=0)  
2              intranet-server CRON[23064]: pam_unix(cron:session): session closed for user root  
3   intranet-server CRON[23137]: pam_unix(cron:session): 

## 2. Find all non-CRON jobs

Let's look at all of the non-CRON jobs, which will contain the accesses by the compromised account

In [11]:
# Assuming 'log_table' is the DataFrame generated from the previous script


def get_interactive_sessions(df):
    # Filter out rows where the 'Log' column contains the string 'CRON'
    # na=False ensures we don't get errors if there are empty rows
    # case=True makes it case-sensitive, though you can set it to False to catch 'cron' as well
    non_cron_df = df[~df["Log"].str.contains("CRON", case=False, na=False)]

    return non_cron_df


# Extract the filtered data
interactive_logs = get_interactive_sessions(log_table)

# Display the results
print("Non-CRON Access Sessions:")
print("-" * 50)
print(interactive_logs.to_string(index=False))

Non-CRON Access Sessions:
--------------------------------------------------
Month Day     Time                                                                                                                                                                Log
  Jan  23 16:23:04                                                                                 intranet-server sshd[15014]: pam_unix(sshd:session): session closed for user jhall
  Jan  23 16:23:04                                                                                                          intranet-server systemd-logind[957]: Removed session 111.
  Jan  23 16:30:46              intranet-server sshd[25184]: Accepted publickey for jhall from 172.19.131.174 port 49828 ssh2: RSA SHA256:8wFbiaYPevKS/wYKnePO20v0iymTcrRh4Kr+1uRS1UM
  Jan  23 16:30:46                                                                      intranet-server sshd[25184]: pam_unix(sshd:session): session opened for user jhall by (uid=0)
  Jan  23 16:

### Breakdown of events

1. Normal User Activity (Jan 23)

    16:23:04: The user jhall ends an existing SSH session.

    16:30:46 - 16:30:47: User jhall logs back in successfully via SSH using public key authentication from the IP address 172.19.131.174.

2. Suspicious SSH Activity (Jan 24)

    03:56:47: There is an anomalous SSH connection attempt from the same IP address (172.19.131.174) that fails because it does not send the proper identification string. This is occasionally seen during automated port scans or broken script executions.

3. Web Server Compromise and Lateral Movement (Jan 24)

    04:37:40: The web server account (www-data) successfully uses the su (substitute user) command to switch its identity to the user jhall. This is highly irregular and strongly suggests an attacker gained access to the web server (likely through a vulnerability in the WordPress site) and used it to pivot to a standard user account.

4. Privilege Escalation and Data Theft (Jan 24)

    04:37:58: The attacker, now operating as jhall, uses sudo (running commands as the root administrator) to execute a list command. The working directory is /var/www/intranet.smith.russellmitchell.com/wp-content/uploads/2022/01, which further points to a compromised WordPress uploads folder acting as the entry point (e.g., a webshell).

    04:38:06: The attacker uses sudo to run /bin/cat /etc/shadow. The /etc/shadow file contains the system's encrypted password hashes. By reading this file, the attacker aims to steal the password hashes to crack them offline.

Conclusion:
These logs describe a classic attack path: an initial compromise via a web directory, lateral movement to a user account (jhall), and successful privilege escalation to access highly sensitive system credentials (/etc/shadow).


## 3. Transform into a DataFrame

Now we'll convert all logs into a structure table (Pandas DataFrame).  
This is exactly what we'll store in our PostgreSQL `auth_logs` table.

In [20]:
import pandas as pd


def parse_auth_log_to_db_schema(file_path):
    # Advanced Regex Pattern Breakdown:
    # Group 1-3: Month, Day, Time (e.g., 'Jan', '23', '16:30:46')
    # Group 4: Hostname (e.g., 'intranet-server')
    # Group 5: Process Name (e.g., 'sshd', 'CRON', 'sudo')
    # Group 6: PID inside brackets (Optional, e.g., '25184')
    # Group 7: Message (Everything after the colon and space)
    log_pattern = re.compile(
        r"^([A-Z][a-z]{2})\s+(\d+)\s+(\d{2}:\d{2}:\d{2})\s+(\S+)\s+([^:\[]+)(?:\[(\d+)\])?:\s+(.*)$"
    )

    parsed_data = []

    # Syslog doesn't include the year, so we must define the current/target year
    # to create a valid PostgreSQL TIMESTAMP.
    target_year = 2024

    try:
        with open(file_path) as file:
            for line in file:
                line = line.strip()

                # Skip empty lines and artifacts
                if not line or line.startswith("["):
                    continue

                match = log_pattern.search(line)
                if match:
                    month, day, time, hostname, process_name, pid, message = match.groups()

                    # Construct a standard timestamp string (e.g., "2024-Jan-23 16:30:46")
                    timestamp_str = f"{target_year}-{month}-{int(day):02d} {time}"
                    # Convert to a Pandas DateTime object
                    event_timestamp = pd.to_datetime(timestamp_str, format="%Y-%b-%d %H:%M:%S")

                    parsed_data.append(
                        {
                            "event_timestamp": event_timestamp,
                            "hostname": hostname,
                            "process_name": process_name.strip(),
                            # Convert PID to an integer if it exists, otherwise keep it as None (NULL in SQL)
                            "pid": int(pid) if pid else None,
                            "message": message.strip(),
                        }
                    )

    except FileNotFoundError:
        print(f"Error: The file {file_path} was not found.")
        return None

    # Convert to DataFrame
    df = pd.DataFrame(parsed_data)

    # Cast the PID column to pandas 'Int64' type which safely supports integers mixed with NaNs (NULLs)
    df["pid"] = df["pid"].astype("Int64")

    return df


# Execute the function
auth_logs_df = parse_auth_log_to_db_schema(auth_log)

# Display a sample of interactive sessions (filtering out CRON)
interactive_sessions = auth_logs_df[~auth_logs_df["process_name"].str.contains("CRON", case=False)]
print(interactive_sessions.head())

# Export exactly as it needs to go into PostgreSQL
auth_logs_df.to_csv("auth_logs_structured.csv", index=False)

       event_timestamp         hostname    process_name    pid  \
65 2024-01-23 16:23:04  intranet-server            sshd  15014   
66 2024-01-23 16:23:04  intranet-server  systemd-logind    957   
67 2024-01-23 16:30:46  intranet-server            sshd  25184   
68 2024-01-23 16:30:46  intranet-server            sshd  25184   
69 2024-01-23 16:30:47  intranet-server         systemd   <NA>   

                                                        message  
65        pam_unix(sshd:session): session closed for user jhall  
66                                         Removed session 111.  
67  Accepted publickey for jhall from 172.19.131.174 port 49...  
68  pam_unix(sshd:session): session opened for user jhall by...  
69  pam_unix(systemd-user:session): session opened for user ...  


In [22]:
# Display the full table
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_columns", None)
auth_logs_df

,event_timestamp,hostname,process_name,pid,message
0,2024-01-23 06:25:05,intranet-server,CRON,22883,pam_unix(cron:session): session closed for user root
1,2024-01-23 06:39:01,intranet-server,CRON,23064,pam_unix(cron:session): session opened for user root by ...
2,2024-01-23 06:39:01,intranet-server,CRON,23064,pam_unix(cron:session): session closed for user root
3,2024-01-23 06:47:01,intranet-server,CRON,23137,pam_unix(cron:session): session opened for user root by ...
4,2024-01-23 06:47:02,intranet-server,CRON,23137,pam_unix(cron:session): session closed for user root
...,...,...,...,...,...
267,2024-01-24 23:09:01,intranet-server,CRON,31372,pam_unix(cron:session): session closed for user root
268,2024-01-24 23:17:01,intranet-server,CRON,31443,pam_unix(cron:session): session opened for user root by ...
269,2024-01-24 23:17:01,intranet-server,CRON,31443,pam_unix(cron:session): session closed for user root
270,2024-01-24 23:39:01,intranet-server,CRON,31519,pam_unix(cron:session): session opened for user root by ...


In [39]:
# Query the DataFrame with SQL (pandasql runs SQL on the in-memory table)

from pandasql import sqldf

sqldf(
    "SELECT event_timestamp, process_name, pid, message FROM auth_logs_df WHERE process_name != 'CRON' ORDER BY event_timestamp ASC"
)

,event_timestamp,process_name,pid,message
0,2024-01-23 16:23:04.000000,sshd,15014.0,pam_unix(sshd:session): session closed for user jhall
1,2024-01-23 16:23:04.000000,systemd-logind,957.0,Removed session 111.
2,2024-01-23 16:30:46.000000,sshd,25184.0,Accepted publickey for jhall from 172.19.131.174 port 49...
3,2024-01-23 16:30:46.000000,sshd,25184.0,pam_unix(sshd:session): session opened for user jhall by...
4,2024-01-23 16:30:47.000000,systemd,NaN,pam_unix(systemd-user:session): session opened for user ...
5,2024-01-23 16:30:47.000000,systemd-logind,957.0,New session 271 of user jhall.
6,2024-01-24 03:56:47.000000,sshd,27751.0,Did not receive identification string from 172.19.131.17...
7,2024-01-24 04:37:40.000000,su,27950.0,Successful su for jhall by www-data
8,2024-01-24 04:37:40.000000,su,27950.0,+ /dev/pts/1 www-data:jhall
9,2024-01-24 04:37:40.000000,su,27950.0,pam_unix(su:session): session opened for user jhall by (...


## 4. Find all priveledge escalations

These indicate suspicious activity

In [34]:
sqldf(
    "SELECT event_timestamp, process_name, message FROM auth_logs_df WHERE process_name IN ('sudo', 'su') ORDER BY event_timestamp"
)

,event_timestamp,process_name,message
0,2024-01-24 04:37:40.000000,su,Successful su for jhall by www-data
1,2024-01-24 04:37:40.000000,su,+ /dev/pts/1 www-data:jhall
2,2024-01-24 04:37:40.000000,su,pam_unix(su:session): session opened for user jhall by (...
3,2024-01-24 04:37:58.000000,sudo,jhall : TTY=pts/1 ; PWD=/var/www/intranet.smith.russellm...
4,2024-01-24 04:38:06.000000,sudo,jhall : TTY=pts/1 ; PWD=/var/www/intranet.smith.russellm...
5,2024-01-24 04:38:06.000000,sudo,pam_unix(sudo:session): session opened for user root by ...
6,2024-01-24 04:38:06.000000,sudo,pam_unix(sudo:session): session closed for user root


## 5. Track SSH connection attempts from specific IPs

172.19.131.174 is the likely compromised address

In [37]:
sqldf(
    "SELECT event_timestamp, message FROM auth_logs_df WHERE process_name = 'sshd' AND message LIKE '%172.19.131.174%'"
)

,event_timestamp,message
0,2024-01-23 16:30:46.000000,Accepted publickey for jhall from 172.19.131.174 port 49...
1,2024-01-24 03:56:47.000000,Did not receive identification string from 172.19.131.17...


## 6. Summarize and count by process name


In [41]:
sqldf(
    "SELECT process_name, COUNT(*) as total_logs FROM auth_logs_df GROUP BY process_name ORDER BY total_logs DESC"
)

,process_name,total_logs
0,CRON,257
1,sudo,4
2,sshd,4
3,systemd-logind,3
4,su,3
5,systemd,1


## 8. Mapping to the Database Schema

Here's how this log data maps to our planned **`hosts`** table in PostgreSQL:

| LOG field | DB Column | SQL Type | Notes |
|-----------|-----------|----------|-------|
| *(auto-generated)* | `auth_log_id` | `SERIAL PRIMARY KEY` | Auto-incrementing ID |
| `event_timestamp` | `event_timestamp` | `VARCHAR(255) NOT NULL` | timestamp of the event|
| `hostname` | `hostname` | `VARCHAR(45)` | host |
| `process_name` | `process_name` | `VARCHAR(50) NOT NULL` | Derived: CRON, sudo, sshd, systemd=logind, su, systemd |
| `message` | `message` | `VARCHAR(255)` | Log message |
| *(auto-generated)* | `created_at` | `TIMESTAMP DEFAULT CURRENT_TIMESTAMP` | When this row was inserted |

### The SQL `CREATE TABLE` statement:

```sql
CREATE TABLE hosts (
    auth_log_id SERIAL PRIMARY KEY,
    event_timestamp VARCHAR(255) NOT NULL UNIQUE,
    hostname VARCHAR(45),
    process_name VARCHAR(50) NOT NULL,
    message VARCHAR(100),
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
```
